# LaBSE fine-tuning on top-5 and bottom-5 **directed pairs**

This notebook implements the corrected protocol implied by:

> each directed pair has ~1000 examples, so top 5 = ~5k datapoints and bottom 5 = ~5k datapoints.

So the training unit is a **directed source-target pair**, not a source language.

- Train top model on 5 top directed pairs from IN22-Gen.
- Train bottom model on 5 bottom directed pairs from IN22-Gen.
- Evaluate base LaBSE, top-pair fine-tuned LaBSE, and bottom-pair fine-tuned LaBSE on held-out IN22-Conv for the same selected directed pairs.


## 1. Install dependencies

Run this first in Colab. After installing, use **Runtime → Restart runtime**, then continue from imports if Colab asks for it.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
%pip -q install -U \
  "sentence-transformers" \
  "datasets" \
  "accelerate" \
  "transformers>=4.51.0,<5" \
  "huggingface_hub" \
  "tqdm" \
  "openpyxl" \
  "pandas==2.2.2" \
  "numpy==2.0.2" \
  "scikit-learn>=1.5,<1.9"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00


## 2. Imports and runtime setup


In [ ]:
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
import gc
import json
import math
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TranslationEvaluator

try:
    from google.colab import drive
    IS_COLAB = True
except Exception:
    IS_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CUDA available: True
GPU: Tesla T4


/tmp/ipykernel_3504/3558718421.py:17: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_3504/3558718421.py:18: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import TranslationEvaluator


## 3. Optional Google Drive output folder

Set `USE_GOOGLE_DRIVE = True` if you want checkpoints/models to persist after the Colab session.


In [ ]:
USE_GOOGLE_DRIVE = IS_COLAB

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/labse_directed_pair_finetuning')
else:
    PROJECT_DIR = Path('./labse_directed_pair_finetuning')

OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project dir:', PROJECT_DIR)
print('Output dir:', OUTPUT_DIR)


Mounted at /content/drive
Project dir: /content/drive/MyDrive/labse_directed_pair_finetuning
Output dir: /content/drive/MyDrive/labse_directed_pair_finetuning/outputs


## 4. Configuration

The key correction is here: `TOP5_DIRECTED_PAIRS` and `BOTTOM5_DIRECTED_PAIRS` are exact directed language pairs.

With `TRAIN_EXAMPLES_PER_DIRECTED_PAIR = 1000`, each group has exactly 5,000 examples before the validation split.


In [ ]:
BASE_MODEL_NAME = 'sentence-transformers/LaBSE'

# Sequence length
MAX_SEQ_LENGTH = 128

# Training
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.10

# Top/bottom directed pairs from pair-level ranking.
TOP5_DIRECTED_PAIRS = [
    ('kan', 'tel'),
    ('urd', 'guj'),
    ('ben', 'kan'),
    ('guj', 'mar'),
    ('guj', 'urd'),
]

BOTTOM5_DIRECTED_PAIRS = [
    ('sat', 'npi'),
    ('sat', 'urd'),
    ('sat', 'hin'),
    ('kan', 'sat'),
    ('ben', 'mni'),
]

RUN_GROUPS = {
    'top5_directed_pairs': TOP5_DIRECTED_PAIRS,
    'bottom5_directed_pairs': BOTTOM5_DIRECTED_PAIRS,
}

# 5 directed pairs × 1000 examples = 5000 examples per group
TRAIN_EXAMPLES_PER_DIRECTED_PAIR = 1000

# Validation split for checkpoint selection and sanity checking
VAL_SIZE = 0.05
VAL_MAX_N = 1000

# Keep 0 for real run
QUICK_TRAIN_N_PER_GROUP = 0
QUICK_EVAL_N_PER_LANGUAGE = 0

# Mixed precision saves GPU memory
USE_AMP = torch.cuda.is_available()

# Evaluation encoding batch size
EVAL_BATCH_SIZE = 128

# Checkpointing
RESUME_FROM_LATEST_CHECKPOINT = False
CHECKPOINTS_PER_EPOCH = 2
MIN_CHECKPOINT_STEPS = 50
CHECKPOINT_SAVE_TOTAL_LIMIT = 20

# Evaluation protocol: train on IN22-Gen, evaluate on IN22-Conv
EVAL_DIRECTED_PAIRS = TOP5_DIRECTED_PAIRS + BOTTOM5_DIRECTED_PAIRS

print('Base model:', BASE_MODEL_NAME)
print('Top 5 directed pairs:', TOP5_DIRECTED_PAIRS)
print('Bottom 5 directed pairs:', BOTTOM5_DIRECTED_PAIRS)
print('Examples per directed pair:', TRAIN_EXAMPLES_PER_DIRECTED_PAIR)

for name, pairs in RUN_GROUPS.items():
    expected = len(pairs) * (TRAIN_EXAMPLES_PER_DIRECTED_PAIR or 1024)
    print(f'{name}: {len(pairs)} directions, approx examples = {expected:,}')

Base model: sentence-transformers/LaBSE
Top 5 directed pairs: [('kan', 'tel'), ('urd', 'guj'), ('ben', 'kan'), ('guj', 'mar'), ('guj', 'urd')]
Bottom 5 directed pairs: [('sat', 'npi'), ('sat', 'urd'), ('sat', 'hin'), ('kan', 'sat'), ('ben', 'mni')]
Examples per directed pair: 1000
top5_directed_pairs: 5 directions, approx examples = 5,000
bottom5_directed_pairs: 5 directions, approx examples = 5,000


## 5. Language mapping helpers


In [ ]:
LANG_TO_IN22 = {
    'asm': 'asm_Beng',
    'ben': 'ben_Beng',
    'brx': 'brx_Deva',
    'doi': 'doi_Deva',
    'gom': 'gom_Deva',
    'guj': 'guj_Gujr',
    'hin': 'hin_Deva',
    'kan': 'kan_Knda',
    'kas': 'kas_Arab',
    'mai': 'mai_Deva',
    'mal': 'mal_Mlym',
    'mar': 'mar_Deva',
    'mni': 'mni_Mtei',
    'npi': 'npi_Deva',
    'ory': 'ory_Orya',
    'pan': 'pan_Guru',
    'san': 'san_Deva',
    'sat': 'sat_Olck',

    # IN22 may use snd_Deva or snd_Arab depending on version.
    # Resolver below checks both.
    'snd': 'snd_Deva',

    'tam': 'tam_Taml',
    'tel': 'tel_Telu',
    'urd': 'urd_Arab',
    'eng': 'eng_Latn',
}

INDIC_LANGS = [
    'asm', 'ben', 'brx', 'doi', 'gom', 'guj', 'hin', 'kan', 'kas', 'mai', 'mal',
    'mar', 'mni', 'npi', 'ory', 'pan', 'san', 'sat', 'snd', 'tam', 'tel', 'urd'
]


def direction_name(src, tgt):
    return f'{src}→{tgt}'


def direction_label(src, tgt):
    return f'{src} → {tgt}'


def resolve_sentence_col(short_lang, df):
    """
    Finds the correct column name for each language.

    Supports both formats:
      hin_Deva
      sentence_hin_Deva

    Also handles Sindhi variants:
      snd_Deva
      snd_Arab
    """

    if short_lang not in LANG_TO_IN22:
        raise KeyError(f'Unknown language code: {short_lang}')

    code = LANG_TO_IN22[short_lang]

    candidates = [
        code,
        f'sentence_{code}',
        short_lang,
        f'sentence_{short_lang}',
    ]

    if short_lang == 'snd':
        candidates += [
            'snd_Deva',
            'sentence_snd_Deva',
            'snd_Arab',
            'sentence_snd_Arab',
        ]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        f'No matching column found for language={short_lang}, code={code}.\n'
        f'Tried: {candidates}\n'
        f'Available columns:\n{list(df.columns)}'
    )


def build_sentence_col_map(df, selected_langs):
    col_map = {}

    for lang in selected_langs:
        col_map[lang] = resolve_sentence_col(lang, df)

    return col_map


# This will be filled after loading the dataframe in Cell 3.
SENTENCE_COLS = {}


def sentence_col(short_lang):
    """
    Returns the resolved real dataframe column name.
    This works only after SENTENCE_COLS is created in Cell 3.
    """

    if short_lang not in SENTENCE_COLS:
        raise KeyError(
            f'{short_lang} not found in SENTENCE_COLS. '
            f'Available keys: {list(SENTENCE_COLS.keys())}. '
            'Run the dataset-loading cell first.'
        )

    return SENTENCE_COLS[short_lang]

## 6. Model/checkpoint helpers


In [ ]:
def make_run_dirs(run_name):
    run_base_dir = OUTPUT_DIR / run_name
    data_dir = run_base_dir / 'data'
    checkpoint_dir = run_base_dir / 'checkpoints'
    best_model_dir = run_base_dir / 'best_model'
    final_model_dir = run_base_dir / 'final_model'

    for p in [run_base_dir, data_dir, checkpoint_dir, best_model_dir, final_model_dir]:
        p.mkdir(parents=True, exist_ok=True)

    return {
        'run_base_dir': run_base_dir,
        'data_dir': data_dir,
        'checkpoint_dir': checkpoint_dir,
        'best_model_dir': best_model_dir,
        'final_model_dir': final_model_dir,
    }


def latest_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)

    if not checkpoint_dir.exists():
        return None

    candidates = [p for p in checkpoint_dir.iterdir() if p.is_dir()]

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)

    return candidates[0]


def load_model_for_run(run_name, checkpoint_dir=None):
    model_to_load = BASE_MODEL_NAME

    if RESUME_FROM_LATEST_CHECKPOINT and checkpoint_dir is not None:
        ckpt = latest_checkpoint(checkpoint_dir)

        if ckpt is not None:
            model_to_load = str(ckpt)
            print(f'[{run_name}] Resuming from checkpoint: {model_to_load}')
        else:
            print(f'[{run_name}] No checkpoint found. Starting from base model.')
    else:
        print(f'[{run_name}] Starting from base model.')

    model = SentenceTransformer(model_to_load, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH

    return model, model_to_load

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face token in Colab Secrets first."
    )

login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

Logged in to Hugging Face.


## 7. Load IN22-Gen and build directed-pair training data

> Add blockquote



If `ai4bharat/IN22-Gen` is gated, run:

```python
from huggingface_hub import login
login()
```

and accept the dataset terms on Hugging Face before this cell.


In [ ]:
def load_in22_dataset_as_dataframe(
    dataset_name,
    dataset_config='default',
    preferred_splits=('gen', 'conv', 'test', 'validation', 'train')
):
    try:
        if dataset_config is None or dataset_config == '':
            dataset_obj = load_dataset(dataset_name)
        else:
            dataset_obj = load_dataset(dataset_name, dataset_config)

    except Exception as e:
        raise RuntimeError(
            f'Could not load {dataset_name} with config={dataset_config!r}. '
            f'Original error: {e}'
        ) from e

    if hasattr(dataset_obj, 'keys'):
        available_splits = list(dataset_obj.keys())

        chosen_split = None

        for split in preferred_splits:
            if split in available_splits:
                chosen_split = split
                break

        if chosen_split is None:
            chosen_split = available_splits[0]

        print(f'Available splits for {dataset_name}: {available_splits}')
        print(f'Using split for {dataset_name}: {chosen_split}')

        return dataset_obj[chosen_split].to_pandas()

    return dataset_obj.to_pandas()


def build_training_df_for_directed_pairs(in22_df, directed_pairs, run_name):
    records = []

    for src, tgt in directed_pairs:
        src_col = sentence_col(src)
        tgt_col = sentence_col(tgt)

        missing = [c for c in [src_col, tgt_col] if c not in in22_df.columns]

        if missing:
            raise ValueError(f'Missing columns for {src}->{tgt}: {missing}')

        keep_cols = [src_col, tgt_col]

        if 'id' in in22_df.columns:
            keep_cols = ['id'] + keep_cols

        pair_df = in22_df[keep_cols].copy()

        pair_df = pair_df.rename(
            columns={
                src_col: 'sentence1',
                tgt_col: 'sentence2',
            }
        )

        pair_df['run_group'] = run_name
        pair_df['source_language'] = src
        pair_df['target_language'] = tgt
        pair_df['direction'] = direction_name(src, tgt)
        pair_df['pair'] = direction_label(src, tgt)

        pair_df['sentence1'] = pair_df['sentence1'].astype(str).str.strip()
        pair_df['sentence2'] = pair_df['sentence2'].astype(str).str.strip()

        pair_df = pair_df[
            pair_df['sentence1'].ne('') &
            pair_df['sentence2'].ne('') &
            pair_df['sentence1'].str.lower().ne('nan') &
            pair_df['sentence2'].str.lower().ne('nan')
        ].copy()

        pair_df = pair_df.drop_duplicates(
            subset=['sentence1', 'sentence2']
        ).reset_index(drop=True)

        if TRAIN_EXAMPLES_PER_DIRECTED_PAIR and TRAIN_EXAMPLES_PER_DIRECTED_PAIR > 0:
            pair_df = pair_df.sample(
                n=min(TRAIN_EXAMPLES_PER_DIRECTED_PAIR, len(pair_df)),
                random_state=SEED,
            ).reset_index(drop=True)

        records.append(pair_df)

    if not records:
        raise ValueError('No training records were created. Check RUN_GROUPS and language columns.')

    out = pd.concat(records, ignore_index=True)

    if QUICK_TRAIN_N_PER_GROUP and QUICK_TRAIN_N_PER_GROUP > 0:
        out = out.sample(
            n=min(QUICK_TRAIN_N_PER_GROUP, len(out)),
            random_state=SEED
        ).reset_index(drop=True)

    return out


# Optional manual login if needed:
# from huggingface_hub import login
# login()


# 1. Load IN22-Gen
in22_gen_df_raw = load_in22_dataset_as_dataframe(
    'ai4bharat/IN22-Gen',
    'default',
    preferred_splits=('gen', 'train', 'test')
)

print('Raw IN22-Gen dataframe shape:', in22_gen_df_raw.shape)

print('\nAvailable dataframe columns:')
print(in22_gen_df_raw.columns.tolist())


# 2. Find all languages needed by RUN_GROUPS
needed_langs = sorted(set([
    lang
    for pairs in RUN_GROUPS.values()
    for pair in pairs
    for lang in pair
]))

print('\nNeeded languages:')
print(needed_langs)


# 3. Resolve actual dataframe columns
SENTENCE_COLS = build_sentence_col_map(
    df=in22_gen_df_raw,
    selected_langs=needed_langs
)

print('\nResolved sentence columns:')
for lang, col in SENTENCE_COLS.items():
    print(f'{lang:4s} -> {col}')


# 4. Build training dataframe for each run group
train_df_by_group = {}

for run_name, directed_pairs in RUN_GROUPS.items():
    group_df = build_training_df_for_directed_pairs(
        in22_df=in22_gen_df_raw,
        directed_pairs=directed_pairs,
        run_name=run_name,
    )

    train_df_by_group[run_name] = group_df

    print('\n' + '=' * 80)
    print(run_name)
    print('Total examples:', len(group_df))
    print('Directions:', group_df['direction'].nunique())

    display(
        group_df.groupby(
            ['source_language', 'target_language', 'pair']
        ).size().reset_index(name='n')
    )


# 5. Sanity check
print('\nSanity check:')

for run_name, group_df in train_df_by_group.items():
    expected = (
        len(RUN_GROUPS[run_name]) * TRAIN_EXAMPLES_PER_DIRECTED_PAIR
        if TRAIN_EXAMPLES_PER_DIRECTED_PAIR
        else None
    )

    print(
        run_name,
        '| actual examples:', len(group_df),
        '| expected if capped:', expected
    )

Available splits for ai4bharat/IN22-Gen: ['test']
Using split for ai4bharat/IN22-Gen: test
Raw IN22-Gen dataframe shape: (1024, 29)

Available dataframe columns:
['context', 'source', 'url', 'domain', 'num_words', 'bucket', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']

Needed languages:
['ben', 'guj', 'hin', 'kan', 'mar', 'mni', 'npi', 'sat', 'tel', 'urd']

Resolved sentence columns:
ben  -> ben_Beng
guj  -> guj_Gujr
hin  -> hin_Deva
kan  -> kan_Knda
mar  -> mar_Deva
mni  -> mni_Mtei
npi  -> npi_Deva
sat  -> sat_Olck
tel  -> tel_Telu
urd  -> urd_Arab

top5_directed_pairs
Total examples: 5000
Directions: 5


,source_language,target_language,pair,n
0,ben,kan,ben → kan,1000
1,guj,mar,guj → mar,1000
2,guj,urd,guj → urd,1000
3,kan,tel,kan → tel,1000
4,urd,guj,urd → guj,1000



bottom5_directed_pairs
Total examples: 5000
Directions: 5


,source_language,target_language,pair,n
0,ben,mni,ben → mni,1000
1,kan,sat,kan → sat,1000
2,sat,hin,sat → hin,1000
3,sat,npi,sat → npi,1000
4,sat,urd,sat → urd,1000



Sanity check:
top5_directed_pairs | actual examples: 5000 | expected if capped: 5000
bottom5_directed_pairs | actual examples: 5000 | expected if capped: 5000


## 8. Train/validation split

The validation split is only a training-time checkpoint sanity check. Final comparison is done on IN22-Conv.


In [ ]:
splits_by_group = {}

for run_name, group_df in train_df_by_group.items():
    if len(group_df) < 100:
        raise ValueError(f'[{run_name}] Training data is too small.')

    stratify_col = group_df['direction'] if group_df['direction'].value_counts().min() >= 2 else None

    train_pairs, val_pairs = train_test_split(
        group_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_col,
    )

    train_pairs = train_pairs.reset_index(drop=True)
    val_pairs = val_pairs.reset_index(drop=True)

    if VAL_MAX_N and len(val_pairs) > VAL_MAX_N:
        val_pairs = val_pairs.sample(n=VAL_MAX_N, random_state=SEED).reset_index(drop=True)

    run_dirs = make_run_dirs(run_name)
    train_pairs_path = run_dirs['data_dir'] / 'train_pairs_used.csv'
    val_pairs_path = run_dirs['data_dir'] / 'val_pairs_used.csv'
    train_pairs.to_csv(train_pairs_path, index=False)
    val_pairs.to_csv(val_pairs_path, index=False)

    splits_by_group[run_name] = {
        'train_pairs': train_pairs,
        'val_pairs': val_pairs,
        'run_dirs': run_dirs,
        'train_pairs_path': train_pairs_path,
        'val_pairs_path': val_pairs_path,
    }

    print('' + '=' * 80)
    print(run_name)
    print('Train examples:', len(train_pairs))
    print('Validation examples:', len(val_pairs))
    display(train_pairs.groupby(['source_language', 'target_language']).size().reset_index(name='train_n'))


top5_directed_pairs
Train examples: 4900
Validation examples: 100


,source_language,target_language,train_n
0,ben,kan,980
1,guj,mar,980
2,guj,urd,980
3,kan,tel,980
4,urd,guj,980


bottom5_directed_pairs
Train examples: 4900
Validation examples: 100


,source_language,target_language,train_n
0,ben,mni,980
1,kan,sat,980
2,sat,hin,980
3,sat,npi,980
4,sat,urd,980


## 9. Dataloaders and evaluator helpers


In [ ]:
def make_train_dataloader(train_pairs):
    train_examples = [
        InputExample(texts=[row['sentence1'], row['sentence2']])
        for _, row in train_pairs.iterrows()
    ]
    return train_examples, DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE, drop_last=True)


def make_validation_evaluator(val_pairs, run_name):
    if len(val_pairs) == 0:
        return None
    return TranslationEvaluator(
        source_sentences=val_pairs['sentence1'].tolist(),
        target_sentences=val_pairs['sentence2'].tolist(),
        name=f'{run_name}_val_translation_retrieval',
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
    )

for run_name, bundle in splits_by_group.items():
    train_examples, train_dataloader = make_train_dataloader(bundle['train_pairs'])
    print('' + '=' * 80)
    print(run_name)
    print('Train examples:', len(train_examples))
    print('Train batches:', len(train_dataloader))
    print('Warmup steps:', math.ceil(len(train_dataloader) * EPOCHS * WARMUP_RATIO))


top5_directed_pairs
Train examples: 4900
Train batches: 153
Warmup steps: 306
bottom5_directed_pairs
Train examples: 4900
Train batches: 153
Warmup steps: 306


## 10. Fine-tune two separate LaBSE models

This trains:

1. `top5_directed_pairs`
2. `bottom5_directed_pairs`

Both start from base LaBSE unless resuming from checkpoints.


In [ ]:
training_summaries = []

for run_name, bundle in splits_by_group.items():
    print('' + '#' * 100)
    print(f'STARTING RUN: {run_name}')
    print('#' * 100)

    run_dirs = bundle['run_dirs']
    train_pairs = bundle['train_pairs']
    val_pairs = bundle['val_pairs']

    model, model_to_load = load_model_for_run(run_name, run_dirs['checkpoint_dir'])
    train_examples, train_dataloader = make_train_dataloader(train_pairs)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    evaluator = make_validation_evaluator(val_pairs, run_name)

    steps_per_epoch = len(train_dataloader)
    if steps_per_epoch == 0:
        raise ValueError(f'[{run_name}] No training batches found. Reduce BATCH_SIZE or add data.')

    warmup_steps = math.ceil(steps_per_epoch * EPOCHS * WARMUP_RATIO)
    checkpoint_save_steps = max(1, steps_per_epoch // CHECKPOINTS_PER_EPOCH)
    checkpoint_save_steps = max(MIN_CHECKPOINT_STEPS, checkpoint_save_steps)
    checkpoint_save_steps = min(checkpoint_save_steps, steps_per_epoch)
    evaluation_steps = checkpoint_save_steps

    training_config = {
        'run_name': run_name,
        'directed_pairs': [list(p) for p in RUN_GROUPS[run_name]],
        'base_model_name': BASE_MODEL_NAME,
        'loaded_model': model_to_load,
        'train_dataset': 'ai4bharat/IN22-Gen',
        'train_dataset_config': 'all',
        'train_examples_per_directed_pair': TRAIN_EXAMPLES_PER_DIRECTED_PAIR,
        'max_seq_length': MAX_SEQ_LENGTH,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'warmup_steps': warmup_steps,
        'steps_per_epoch': steps_per_epoch,
        'evaluation_steps': evaluation_steps,
        'checkpoint_save_steps': checkpoint_save_steps,
        'checkpoint_save_total_limit': CHECKPOINT_SAVE_TOTAL_LIMIT,
        'train_pairs': len(train_pairs),
        'validation_pairs': len(val_pairs),
        'use_amp': USE_AMP,
        'final_model_dir': str(run_dirs['final_model_dir']),
        'best_model_dir': str(run_dirs['best_model_dir']),
        'checkpoint_dir': str(run_dirs['checkpoint_dir']),
        'train_pairs_path': str(bundle['train_pairs_path']),
        'val_pairs_path': str(bundle['val_pairs_path']),
    }

    config_path = run_dirs['run_base_dir'] / 'training_config.json'
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(training_config, f, indent=2, ensure_ascii=False)

    print('Run:', run_name)
    print('Directed pairs:', [direction_label(*p) for p in RUN_GROUPS[run_name]])
    print('Train examples:', len(train_pairs))
    print('Validation examples:', len(val_pairs))
    print('Steps per epoch:', steps_per_epoch)
    print('Evaluation/checkpoint steps:', evaluation_steps)
    print('Best model folder:', run_dirs['best_model_dir'])
    print('Final model folder:', run_dirs['final_model_dir'])

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=evaluator,
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        output_path=str(run_dirs['best_model_dir']),
        optimizer_params={'lr': LEARNING_RATE},
        evaluation_steps=evaluation_steps,
        save_best_model=True,
        show_progress_bar=True,
        use_amp=True,
        checkpoint_path=str(run_dirs['checkpoint_dir']),
        checkpoint_save_steps=checkpoint_save_steps,
        checkpoint_save_total_limit=CHECKPOINT_SAVE_TOTAL_LIMIT,
    )

    model.save(str(run_dirs['final_model_dir']))
    training_summaries.append(training_config)

    print(f'[{run_name}] Saved final model to: {run_dirs["final_model_dir"]}')
    print(f'[{run_name}] Best validation model at: {run_dirs["best_model_dir"]}')

    del model, train_loss, train_dataloader, train_examples, evaluator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

training_summary_path = OUTPUT_DIR / 'training_summaries_directed_pairs.json'
with open(training_summary_path, 'w', encoding='utf-8') as f:
    json.dump(training_summaries, f, indent=2, ensure_ascii=False)
print('Training summaries saved to:', training_summary_path)


####################################################################################################
STARTING RUN: top5_directed_pairs
####################################################################################################
[top5_directed_pairs] No checkpoint found. Starting from base model.


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

## 11. Output folders


In [ ]:
for run_name in RUN_GROUPS:
    run_dirs = make_run_dirs(run_name)
    print('' + '=' * 80)
    print(run_name)
    print('Final model:', run_dirs['final_model_dir'])
    print('Best validation model:', run_dirs['best_model_dir'])
    print('Checkpoints:', run_dirs['checkpoint_dir'])
    print('Data:', run_dirs['data_dir'])


# Evaluation on IN22-Conv

This section evaluates base LaBSE and both fine-tuned models on held-out IN22-Conv for the same 10 selected directed pairs.


## 12. Load and clean IN22-Conv


In [ ]:
def clean_eval_dataframe(eval_df, eval_langs):
    eval_df = eval_df.copy()
    needed_cols = [sentence_col(lang) for lang in eval_langs]
    missing_cols = [col for col in needed_cols if col not in eval_df.columns]
    if missing_cols:
        raise ValueError(f'Missing expected sentence columns in eval dataset: {missing_cols}')

    for col in needed_cols:
        eval_df[col] = eval_df[col].astype(str).str.strip()

    mask = np.ones(len(eval_df), dtype=bool)
    for col in needed_cols:
        mask &= eval_df[col].ne('')
        mask &= eval_df[col].str.lower().ne('nan')

    eval_df = eval_df.loc[mask].reset_index(drop=True)

    if QUICK_EVAL_N_PER_LANGUAGE and QUICK_EVAL_N_PER_LANGUAGE > 0:
        eval_df = eval_df.sample(
            n=min(QUICK_EVAL_N_PER_LANGUAGE, len(eval_df)),
            random_state=SEED,
        ).reset_index(drop=True)

    return eval_df

conv_df_raw = load_in22_dataset_as_dataframe('ai4bharat/IN22-Conv', 'all', preferred_splits=('conv', 'test', 'validation', 'train'))
eval_langs = sorted(set([lang for pair in EVAL_DIRECTED_PAIRS for lang in pair]))
conv_df = clean_eval_dataframe(conv_df_raw, eval_langs)

print('Raw IN22-Conv shape:', conv_df_raw.shape)
print('Clean IN22-Conv shape:', conv_df.shape)
print('Evaluation languages:', eval_langs)
print('Evaluation directed pairs:', [direction_label(*p) for p in EVAL_DIRECTED_PAIRS])


## 13. Evaluation metric helpers

The evaluation mirrors your pre-test:

- gold cosine = aligned source-target sentence cosine
- random cosine = source sentence vs shifted target sentence
- threshold midpoint = `(mean_gold_cosine + mean_random_cosine) / 2`
- sensitivity = fraction of gold cosine above threshold
- specificity = fraction of random cosine below threshold


In [ ]:
def stable_offset(n, seed, src_lang, tgt_lang, model_name):
    if n < 2:
        raise ValueError('Need at least 2 rows to create random negatives.')
    key = f'{seed}:{src_lang}:{tgt_lang}:{model_name}'.encode('utf-8')
    digest = hashlib.md5(key).hexdigest()
    return (int(digest[:8], 16) % (n - 1)) + 1


def encode_eval_languages(model, eval_df, eval_langs, batch_size=128):
    embeddings_by_lang = {}
    for lang in eval_langs:
        col = sentence_col(lang)
        sentences = eval_df[col].tolist()
        print(f'Encoding {lang} ({col}) | n={len(sentences):,}')
        embeddings = model.encode(
            sentences,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )
        embeddings_by_lang[lang] = embeddings.astype(np.float32, copy=False)
    return embeddings_by_lang


def evaluate_embeddings_for_directed_pairs(embeddings_by_lang, directed_pairs, model_name, dataset_suite):
    records = []
    for src, tgt in directed_pairs:
        src_emb = embeddings_by_lang[src]
        tgt_emb = embeddings_by_lang[tgt]
        if src_emb.shape[0] != tgt_emb.shape[0]:
            raise ValueError(f'Row mismatch for {src}->{tgt}: {src_emb.shape[0]} vs {tgt_emb.shape[0]}')

        n = src_emb.shape[0]
        gold_cos = np.sum(src_emb * tgt_emb, axis=1)

        offset = stable_offset(n, SEED, src, tgt, model_name)
        random_tgt_emb = np.roll(tgt_emb, shift=offset, axis=0)
        random_cos = np.sum(src_emb * random_tgt_emb, axis=1)

        mean_gold = float(np.mean(gold_cos))
        mean_random = float(np.mean(random_cos))
        threshold_midpoint = float((mean_gold + mean_random) / 2.0)
        sensitivity = float(np.mean(gold_cos >= threshold_midpoint))
        specificity = float(np.mean(random_cos < threshold_midpoint))

        pair_group = 'top5_directed_pairs' if (src, tgt) in TOP5_DIRECTED_PAIRS else 'bottom5_directed_pairs'

        records.append({
            'model_name': model_name,
            'dataset_suite': dataset_suite,
            'pair_group': pair_group,
            'source_language': src,
            'target_language': tgt,
            'pair': direction_label(src, tgt),
            'direction': direction_name(src, tgt),
            'n_eval_pairs': int(n),
            'mean_gold_cosine': mean_gold,
            'mean_random_cosine': mean_random,
            'cosine_gap': mean_gold - mean_random,
            'threshold_midpoint': threshold_midpoint,
            'sensitivity_midpoint': sensitivity,
            'specificity_midpoint': specificity,
            'balanced_accuracy_midpoint': (sensitivity + specificity) / 2.0,
        })

    return pd.DataFrame(records)


def evaluate_sentence_transformer_model(mode., model_name, eval_df, eval_langs, directed_pairs):
    print('' + '#' * 100)
    print('Evaluating:', model_name)
    print('Model path/name:', model_path_or_name)
    print('#' * 100)

    model = SentenceTransformer(str(model_path_or_name))
    model.max_seq_length = MAX_SEQ_LENGTH
    embeddings_by_lang = encode_eval_languages(model, eval_df, eval_langs, batch_size=EVAL_BATCH_SIZE)
    results = evaluate_embeddings_for_directed_pairs(
        embeddings_by_lang=embeddings_by_lang,
        directed_pairs=directed_pairs,
        model_name=model_name,
        dataset_suite='IN22-Conv',
    )

    del model, embeddings_by_lang
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


## 14. Evaluate base, top-pair model, and bottom-pair model

By default, this uses the `best_model` folders. Change to `final_model` if needed.


In [ ]:
EVAL_MODEL_VARIANT = 'best'  # 'best' or 'final'

def model_dir_for_eval(run_name, variant='best'):
    run_dirs = make_run_dirs(run_name)
    return run_dirs['best_model_dir'] if variant == 'best' else run_dirs['final_model_dir']

models_to_evaluate = {
    'labse_base': BASE_MODEL_NAME,
}

for run_name in RUN_GROUPS:
    candidate_dir = model_dir_for_eval(run_name, EVAL_MODEL_VARIANT)
    if candidate_dir.exists() and any(candidate_dir.iterdir()):
        models_to_evaluate[f'labse_{run_name}_{EVAL_MODEL_VARIANT}'] = str(candidate_dir)
    else:
        print(f'WARNING: skipping {run_name}; model folder not found or empty: {candidate_dir}')

print('Models to evaluate:')
for name, path in models_to_evaluate.items():
    print(' -', name, '=>', path)

all_eval_results = []
for model_name, model_path in models_to_evaluate.items():
    result_df = evaluate_sentence_transformer_model(
        model_path_or_name=model_path,
        model_name=model_name,
        eval_df=conv_df,
        eval_langs=eval_langs,
        directed_pairs=EVAL_DIRECTED_PAIRS,
    )
    all_eval_results.append(result_df)

conv_eval_by_pair = pd.concat(all_eval_results, ignore_index=True)
conv_eval_by_pair = conv_eval_by_pair.sort_values(['pair_group', 'pair', 'model_name']).reset_index(drop=True)

eval_output_dir = OUTPUT_DIR / 'conv_eval_directed_pairs'
eval_output_dir.mkdir(parents=True, exist_ok=True)
conv_eval_csv = eval_output_dir / 'conv_eval_all_models_by_directed_pair.csv'
conv_eval_by_pair.to_csv(conv_eval_csv, index=False)

print('Saved pair-level eval to:', conv_eval_csv)
display(conv_eval_by_pair)


## 15. Group summaries and delta tables

Use these to answer: did fine-tuning improve the selected top/bottom directed pairs on IN22-Conv?


In [ ]:
def summarize_by_group(eval_df):
    metric_cols = [
        'mean_gold_cosine',
        'mean_random_cosine',
        'cosine_gap',
        'threshold_midpoint',
        'sensitivity_midpoint',
        'specificity_midpoint',
        'balanced_accuracy_midpoint',
    ]
    return (
        eval_df.groupby(['model_name', 'pair_group'], as_index=False)
        .agg({**{c: 'mean' for c in metric_cols}, 'pair': 'count'})
        .rename(columns={'pair': 'num_directed_pairs'})
        .sort_values(['pair_group', 'model_name'])
        .reset_index(drop=True)
    )


def make_delta_by_pair(eval_df, tuned_model_name, baseline_model_name='labse_base'):
    metrics = [
        'mean_gold_cosine',
        'mean_random_cosine',
        'cosine_gap',
        'threshold_midpoint',
        'sensitivity_midpoint',
        'specificity_midpoint',
        'balanced_accuracy_midpoint',
    ]
    base = eval_df[eval_df['model_name'] == baseline_model_name].copy()
    tuned = eval_df[eval_df['model_name'] == tuned_model_name].copy()

    keep = ['pair_group', 'source_language', 'target_language', 'pair', 'direction', 'n_eval_pairs'] + metrics
    base = base[keep].rename(columns={m: f'{m}_baseline' for m in metrics})
    tuned = tuned[keep].rename(columns={m: f'{m}_tuned' for m in metrics})

    merged = tuned.merge(
        base,
        on=['pair_group', 'source_language', 'target_language', 'pair', 'direction', 'n_eval_pairs'],
        how='inner',
    )

    for m in metrics:
        merged[f'delta_{m}'] = merged[f'{m}_tuned'] - merged[f'{m}_baseline']

    return merged.sort_values(['pair_group', 'pair']).reset_index(drop=True)

conv_eval_summary_by_group = summarize_by_group(conv_eval_by_pair)
summary_csv = eval_output_dir / 'conv_eval_summary_by_pair_group.csv'
conv_eval_summary_by_group.to_csv(summary_csv, index=False)
print('Saved group summary to:', summary_csv)
display(conv_eval_summary_by_group)

for run_name in RUN_GROUPS:
    tuned_name = f'labse_{run_name}_{EVAL_MODEL_VARIANT}'
    if tuned_name in conv_eval_by_pair['model_name'].unique():
        delta_df = make_delta_by_pair(conv_eval_by_pair, tuned_model_name=tuned_name, baseline_model_name='labse_base')
        delta_csv = eval_output_dir / f'delta_{tuned_name}_vs_labse_base_by_directed_pair.csv'
        delta_df.to_csv(delta_csv, index=False)
        print('Saved delta table:', delta_csv)
        display(delta_df[[
            'pair_group', 'pair',
            'sensitivity_midpoint_baseline', 'sensitivity_midpoint_tuned', 'delta_sensitivity_midpoint',
            'specificity_midpoint_baseline', 'specificity_midpoint_tuned', 'delta_specificity_midpoint',
            'balanced_accuracy_midpoint_baseline', 'balanced_accuracy_midpoint_tuned', 'delta_balanced_accuracy_midpoint'
        ]])


## 16. Quick interpretation guide

For the advisor's setup, the key rows are:

- `labse_top5_directed_pairs_best` on `pair_group == top5_directed_pairs`
- `labse_bottom5_directed_pairs_best` on `pair_group == bottom5_directed_pairs`

Compare these against `labse_base` using the delta CSVs. Positive `delta_sensitivity_midpoint`, `delta_specificity_midpoint`, or `delta_balanced_accuracy_midpoint` means improvement on held-out IN22-Conv.
